<a href="https://colab.research.google.com/github/Mez-MohamedRafik/Mez-MohamedRafik/blob/main/Customer_churn_RFC_pred.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# dataset link  https://www.kaggle.com/datasets/radheshyamkollipara/bank-customer-churn

In [56]:
# Data Preprocessing & Cleaning Notice
#Prior to modeling in Python, the dataset underwent initial data wrangling, cleaning, and sanitization in MySQL. Key data quality procedures included identifying and removing duplicate records, verifying column data types, handling missing values, and validating data integrity across key feature distributions (e.g., balance, age, active status, and product counts). The cleaned output was exported directly to CleanedChurnModeling.csv to serve as the unified, consistent baseline for exploratory analysis, feature encoding, scaling, and machine learning pipeline training in this notebook.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix

In [3]:
df = pd.read_csv(r"CleanedChurnModeling.csv", sep = ';') # uploaded here in colab
df.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [7]:
df['Exited'].value_counts()
# the dataset is unbalanced

,count
Exited,
0,7887
1,2021


In [11]:
X = df.drop(['RowNumber','CustomerId' ,'Surname','Exited'], axis = 1)
y = df['Exited'].values
X.columns

Index(['CreditScore', 'Geography', 'Gender', 'Age', 'Tenure', 'Balance',
       'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary'],
      dtype='object')

In [14]:
categorical_features = ['Geography', 'Gender']
numerical_features = [col for col in X.columns if col not in categorical_features]
display(numerical_features)

['CreditScore',
 'Age',
 'Tenure',
 'Balance',
 'NumOfProducts',
 'HasCrCard',
 'IsActiveMember',
 'EstimatedSalary']

In [15]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num', StandardScaler(), numerical_features),
        ('cat',OneHotEncoder(drop = 'first', handle_unknown='ignore'), categorical_features)
    ]
)

In [16]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42, class_weight='balanced'))

])

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state=42, stratify=y)


In [18]:
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['CreditScore', 'Age',
                                                   'Tenure', 'Balance',
                                                   'NumOfProducts', 'HasCrCard',
                                                   'IsActiveMember',
                                                   'EstimatedSalary']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['Geography', 'Gender'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced',
                                        random_state=42))])

In [40]:
y_pred = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:,1]
custom_threshold = 0.57
y_pred_adj = (y_proba >= custom_threshold).astype(int)

print("=== Evaluation Metrics ===")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred_adj))

=== Evaluation Metrics ===
ROC-AUC Score: 0.8370

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.97      0.92      1578
           1       0.78      0.42      0.55       404

    accuracy                           0.86      1982
   macro avg       0.82      0.69      0.73      1982
weighted avg       0.85      0.86      0.84      1982



In [42]:
cm = confusion_matrix(y_test, y_pred)
print("Confusion Matrix:")
print(f"True Negative (TN):{cm[0,0]} | False Positive (FP):{cm[0,1]}")
print(f"False Negative (FN):{cm[1,0]} | True Positive (TP):{cm[1,1]}")

Confusion Matrix:
True Negative (TN):1508 | False Positive (FP):70
False Negative (FN):220 | True Positive (TP):184


In [ ]:
# Trying the XGBoost model

In [44]:
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score

xgb_model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42,
    scale_pos_weight=3.9,
    eval_metric='logloss'
)

In [45]:
xgb_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', xgb_model)
])

In [46]:
xgb_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['CreditScore', 'Age',
                                                   'Tenure', 'Balance',
                                                   'NumOfProducts', 'HasCrCard',
                                                   'IsActiveMember',
                                                   'EstimatedSalary']),
                                                 ('cat',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore'),
                                                  ['Geography', 'Gender'])])),
                ('classifier',
                 XGBClassifier(base_score=None, booster=None, callbacks=None...
                               feature_types=None, feature_weights=None,
                               gamma=None, grow_policy=None,
                               importance_type=None,
                               interaction_constraints=None, learning_rate=0.1,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=5, max_leaves=None,
                               min_child_weight=None, missing=nan,
                               monotone_constraints=None, multi_strategy=None,
                               n_estimators=100, n_jobs=None,
                               num_parallel_tree=None, ...))])

In [55]:
y_pred_xgb = xgb_pipeline.predict(X_test)
y_proba_xgb = xgb_pipeline.predict_proba(X_test)[:,1]

custom_threshold2 = 0.35
y_pred_adj_xgb = (y_proba_xgb >= custom_threshold2).astype(int)

print("=== XGBoost Evaluation Metrics ===")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_proba_xgb):.4f}\n")
print(classification_report(y_test, y_pred_adj_xgb))


cm_xgb = confusion_matrix(y_test, y_pred_adj_xgb)
print("Confusion Matrix:")
print(f"TN: {cm_xgb[0,0]} | FP: {cm_xgb[0,1]}")
print(f"FN: {cm_xgb[1,0]} | TP: {cm_xgb[1,1]}")

=== XGBoost Evaluation Metrics ===
ROC-AUC Score: 0.8512

              precision    recall  f1-score   support

           0       0.94      0.70      0.80      1578
           1       0.41      0.82      0.54       404

    accuracy                           0.72      1982
   macro avg       0.67      0.76      0.67      1982
weighted avg       0.83      0.72      0.75      1982

Confusion Matrix:
TN: 1097 | FP: 481
FN: 73 | TP: 331


In [ ]:
# --------------------------------------------------------------------------------
# THRESHOLD SELECTION STRATEGY:
#
# 1. threshold = 0.35 -> MAXIMIZE RETENTION (Minimizes FN / Missed Churners)
#    - Best for: Preventing customer loss at all costs.
#    - Risk: Higher False Positives (safe customers receive unnecessary retention emails).
#
# 2. threshold = 0.50 -> BALANCED BASELINE (Default with scale_pos_weight = 3.9)
#    - Best for: Good balance between catching churners (~73% Recall) and precision.
#
# 3. threshold = 0.60 -> COST CONTROL (Minimizes FP / Wasted Budget)
#    - Best for: When retention offers (discounts/gifts) are expensive.
# --------------------------------------------------------------------------------